<a href="https://colab.research.google.com/github/EconShark/23_big_data_analysis_class/blob/main/Baitap02/Cleaning_Data_with_PySpark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 1. Clone only the repository metadata
!git clone --depth 1 --filter=blob:none --sparse https://github.com/EconShark/23_big_data_analysis_class.git

# 2. Check out only the Lab04 datasets directory
!cd 23_big_data_analysis_class && git sparse-checkout set Lab04_Clearning_Data_with_PySpark/Datasets

# 3. Copy the datasets to /content/Datasets
!cp -rn 23_big_data_analysis_class/Lab04_Clearning_Data_with_PySpark/Datasets/. /content/Datasets/

# 4. Remove the cloned repository
!rm -rf 23_big_data_analysis_class

Cloning into '23_big_data_analysis_class'...
remote: Enumerating objects: 26, done.
remote: Counting objects: 100% (26/26), done.
remote: Compressing objects: 100% (26/26), done.
remote: Total 26 (delta 0), reused 18 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (26/26), 5.80 KiB | 5.80 MiB/s, done.
remote: Enumerating objects: 4, done.
remote: Counting objects: 100% (4/4), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 4 (delta 0), reused 4 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (4/4), 168.31 KiB | 2.24 MiB/s, done.
remote: Enumerating objects: 8, done.
remote: Counting objects: 100% (8/8), done.
remote: Compressing objects: 100% (8/8), done.
remote: Total 8 (delta 0), reused 7 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (8/8), 3.79 MiB | 14.64 MiB/s, done.
cp: warning: behavior of -n is non-portable and may change in future; use --update=none instead


<a id="setup"></a>
## Bài 1. Khởi tạo môi trường PySpark

### Bước 1. Cài PySpark nếu cần

In [2]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("pyspark") is None:
    print("Chưa có PySpark. Đang cài đặt...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyspark"])
else:
    print("PySpark đã sẵn sàng.")

PySpark đã sẵn sàng.


In [3]:
from pathlib import Path

candidates = [
    Path.cwd() / "Datasets",
    Path("/content/Datasets"),
    Path.cwd().parent / "Datasets"
]

DATA_DIR = next((p for p in candidates if p.exists()), None)

if DATA_DIR is None:
    print("Không tìm thấy thư mục Datasets.")
else:
    print("DATA_DIR =", DATA_DIR.resolve())
    for p in sorted(DATA_DIR.iterdir()):
        print(" -", p.name)

DATA_DIR = /content/Datasets
 - AA_DFW_2014_Departures_Short.csv.gz
 - AA_DFW_2015_Departures_Short.csv.gz
 - AA_DFW_2016_Departures_Short.csv.gz
 - AA_DFW_2017_Departures_Short.csv.gz
 - DallasCouncilVoters.csv.gz
 - DallasCouncilVotes.csv.gz


### Bước 2. Khởi tạo SparkSession

In [4]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

spark = (
    SparkSession.builder
    .appName("CleaningDataWithPySpark")
    .master("local[*]")
    .getOrCreate()
)

sc = spark.sparkContext

print("Spark version :", spark.version)
print("Master        :", sc.master)
print("Parallelism   :", sc.defaultParallelism)

Spark version : 4.0.4
Master        : local[*]
Parallelism   : 2


## Bài 2. Data Cleaning và ví dụ cơ bản

Data Cleaning là quá trình chuẩn bị dữ liệu thô trước khi đưa vào pipeline xử lý.

Các công việc thường gặp:

- chuẩn hóa hoặc thay thế chuỗi;
- tính toán lại giá trị;
- loại dữ liệu rác;
- xử lý dữ liệu thiếu;
- kiểm tra kiểu dữ liệu;
- chuẩn hóa định dạng;
- xác thực dữ liệu.

Spark phù hợp cho data cleaning khi dữ liệu có kích thước lớn vì có khả năng xử lý phân tán và mở rộng.

### 2.1. Tạo dữ liệu thô minh họa

Dữ liệu thô:

| name | age (years) | city |
|---|---:|---|
| Smith, John | 37 | Dallas |
| Wilson, A. | 59 | Chicago |
| null | 215 | null |

Mục tiêu:

- tách họ và tên;
- đổi tuổi từ năm sang tháng;
- đổi tên thành phố sang mã bang;
- loại dòng không hợp lệ.

In [5]:
raw_data = [
    ("Smith, John", 37, "Dallas"),
    ("Wilson, A.", 59, "Chicago"),
    (None, 215, None)
]

raw_df = spark.createDataFrame(raw_data, ["name", "age_years", "city"])
raw_df.show(truncate=False)

+-----------+---------+-------+
|name       |age_years|city   |
+-----------+---------+-------+
|Smith, John|37       |Dallas |
|Wilson, A. |59       |Chicago|
|NULL       |215      |NULL   |
+-----------+---------+-------+



### 2.2. Làm sạch dữ liệu
Ta loại dòng không có tên, tách họ/tên, chuyển tuổi từ năm sang tháng và ánh xạ thành phố sang mã
bang.

In [6]:
city_to_state = (
    F.when(F.col("city") == "Dallas", "TX")
     .when(F.col("city") == "Chicago", "IL")
)

clean_demo_df = (
    raw_df
    .filter(F.col("name").isNotNull())
    .withColumn("last_name", F.trim(F.split("name", ",").getItem(0)))
    .withColumn("first_name", F.trim(F.split("name", ",").getItem(1)))
    .withColumn("age_months", F.col("age_years") * 12)
    .withColumn("state", city_to_state)
    .select("last_name", "first_name", "age_months", "state")
)

clean_demo_df.show(truncate=False)

+---------+----------+----------+-----+
|last_name|first_name|age_months|state|
+---------+----------+----------+-----+
|Smith    |John      |444       |TX   |
|Wilson   |A.        |708       |IL   |
+---------+----------+----------+-----+



<a id="schema"></a>
## Bài 3. Spark Schema

Schema mô tả cấu trúc của DataFrame:

- tên cột;
- kiểu dữ liệu;
- có cho phép `null` hay không.

Việc khai báo schema rõ ràng giúp:

- phát hiện dữ liệu sai kiểu sớm hơn;
- tránh Spark phải quét dữ liệu nhiều lần để suy luận kiểu;
- tăng tính nhất quán trong pipeline.

### 3.1. Schema cho dữ liệu chuyến bay

In [8]:
flight_schema = StructType([
    StructField("Date (MM/DD/YYYY)", StringType(), True),
    StructField("Flight Number", StringType(), True),
    StructField("Destination Airport", StringType(), True),
    StructField("Actual elapsed time (Minutes)", IntegerType(), True)
])


flight_2014 = spark.read.csv(
    str(DATA_DIR / "AA_DFW_2014_Departures_Short.csv.gz"),
    header=True,
    schema=flight_schema
)

flight_2014.printSchema()
flight_2014.show(5)

root
 |-- Date (MM/DD/YYYY): string (nullable = true)
 |-- Flight Number: string (nullable = true)
 |-- Destination Airport: string (nullable = true)
 |-- Actual elapsed time (Minutes): integer (nullable = true)

+-----------------+-------------+-------------------+-----------------------------+
|Date (MM/DD/YYYY)|Flight Number|Destination Airport|Actual elapsed time (Minutes)|
+-----------------+-------------+-------------------+-----------------------------+
|       01/01/2014|         0005|                HNL|                          519|
|       01/01/2014|         0007|                OGG|                          505|
|       01/01/2014|         0035|                SLC|                          174|
|       01/01/2014|         0043|                DTW|                          153|
|       01/01/2014|         0052|                PIT|                          137|
+-----------------+-------------+-------------------+-----------------------------+
only showing top 5 rows


Nếu dữ liệu có kích thước lớn và cấu trúc đã biết, nên ưu tiên schema tường minh thay vì
inferSchema=True

## Bài 4. Immutability và Lazy Processing

### 4.1. Immutability

Spark DataFrame là immutable.  
Transformation không sửa trực tiếp DataFrame cũ mà tạo một DataFrame logic mới.

In [9]:
df_original = spark.createDataFrame([
    ("Alice", 24),
    ("Bob", 30)
], ["name", "age"])

df_new = df_original.withColumn("age_next_year", F.col("age") + 1)

print("DataFrame gốc:")
df_original.show()

print("DataFrame mới:")
df_new.show()

DataFrame gốc:
+-----+---+
| name|age|
+-----+---+
|Alice| 24|
|  Bob| 30|
+-----+---+

DataFrame mới:
+-----+---+-------------+
| name|age|age_next_year|
+-----+---+-------------+
|Alice| 24|           25|
|  Bob| 30|           31|
+-----+---+-------------+



### 4.2. Lazy Processing

Các lệnh:

- `withColumn()`
- `select()`
- `filter()`
- `drop()`

là transformations.

Spark trì hoãn thực thi đến khi có **action** như:

- `show()`
- `count()`
- `collect()`
- `write`

In [10]:
lazy_df = flight_2014.withColumn(
  "airport",
  F.lower(F.col("Destination Airport"))
).drop("Destination Airport")

# Action
lazy_df.show(5)

+-----------------+-------------+-----------------------------+-------+
|Date (MM/DD/YYYY)|Flight Number|Actual elapsed time (Minutes)|airport|
+-----------------+-------------+-----------------------------+-------+
|       01/01/2014|         0005|                          519|    hnl|
|       01/01/2014|         0007|                          505|    ogg|
|       01/01/2014|         0035|                          174|    slc|
|       01/01/2014|         0043|                          153|    dtw|
|       01/01/2014|         0052|                          137|    pit|
+-----------------+-------------+-----------------------------+-------+
only showing top 5 rows


### 4.3. Quan sát kế hoạch thực thi

In [11]:
lazy_df.filter(F.col("airport") == "hnl").select("airport").explain()

== Physical Plan ==
*(1) Project [lower(Destination Airport#32) AS airport#72]
+- *(1) Filter (isnotnull(Destination Airport#32) AND (lower(Destination Airport#32) = hnl))
   +- FileScan csv [Destination Airport#32] Batched: false, DataFilters: [isnotnull(Destination Airport#32), (lower(Destination Airport#32) = hnl)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/Datasets/AA_DFW_2014_Departures_Short.csv.gz], PartitionFilters: [], PushedFilters: [IsNotNull(Destination Airport)], ReadSchema: struct<Destination Airport:string>




## Bài 5. Làm việc với Parquet

CSV dễ trao đổi nhưng có các hạn chế:

- không lưu schema;
- cần parse chuỗi;
- kém hiệu quả với dữ liệu lớn;
- hạn chế predicate pushdown.

Parquet là định dạng dạng cột (columnar), phù hợp với Spark vì:

- lưu schema;
- hỗ trợ predicate pushdown;
- đọc chọn lọc cột hiệu quả;
- thường tốt hơn cho phân tích.

Đây là định dạng phù hợp cho dữ liệu trung gian trong nhiều pipeline Spark.

### 5.1. Gộp nhiều năm dữ liệu

In [12]:
files = [
    "AA_DFW_2014_Departures_Short.csv.gz",
    "AA_DFW_2015_Departures_Short.csv.gz",
    "AA_DFW_2016_Departures_Short.csv.gz",
    "AA_DFW_2017_Departures_Short.csv.gz"
]

flight_dfs = [
    spark.read.csv(str(DATA_DIR / f), header=True, schema=flight_schema)
    for f in files
]

flights_all = flight_dfs[0]
for df in flight_dfs[1:]:
    flights_all = flights_all.unionByName(df)

print("Tổng số dòng:", flights_all.count())
flights_all.show(5)

Tổng số dòng: 583718
+-----------------+-------------+-------------------+-----------------------------+
|Date (MM/DD/YYYY)|Flight Number|Destination Airport|Actual elapsed time (Minutes)|
+-----------------+-------------+-------------------+-----------------------------+
|       01/01/2014|         0005|                HNL|                          519|
|       01/01/2014|         0007|                OGG|                          505|
|       01/01/2014|         0035|                SLC|                          174|
|       01/01/2014|         0043|                DTW|                          153|
|       01/01/2014|         0052|                PIT|                          137|
+-----------------+-------------+-------------------+-----------------------------+
only showing top 5 rows


### 5.2. Ghi và đọc lại Parquet

In [13]:
PARQUET_PATH = str(Path.cwd() / "AA_DFW_ALL.parquet")

(
    flights_all.write
    .mode("overwrite")
    .parquet(PARQUET_PATH)
)

flights_parquet = spark.read.parquet(PARQUET_PATH)

print("Đọc lại từ Parquet:", flights_parquet.count())
flights_parquet.printSchema()

Đọc lại từ Parquet: 583718
root
 |-- Date (MM/DD/YYYY): string (nullable = true)
 |-- Flight Number: string (nullable = true)
 |-- Destination Airport: string (nullable = true)
 |-- Actual elapsed time (Minutes): integer (nullable = true)



## Bài 6. DataFrame Column Operations

Các thao tác cột là phần cốt lõi của data cleaning trong PySpark.

Các transformation phổ biến:

- `filter()` / `where()`
- `select()`
- `withColumn()`
- `drop()`
- `withColumnRenamed()`

### 6.1. Đọc dữ liệu Dallas Council Voters

In [14]:
voter_df = spark.read.csv(
    str(DATA_DIR / "DallasCouncilVoters.csv.gz"),
    header=True,
    inferSchema=True
)

voter_df.printSchema()
voter_df.show(10, truncate=False)

root
 |-- DATE: string (nullable = true)
 |-- TITLE: string (nullable = true)
 |-- VOTER_NAME: string (nullable = true)

+----------+-------------+-------------------+
|DATE      |TITLE        |VOTER_NAME         |
+----------+-------------+-------------------+
|02/08/2017|Councilmember|Jennifer S. Gates  |
|02/08/2017|Councilmember|Philip T. Kingston |
|02/08/2017|Mayor        |Michael S. Rawlings|
|02/08/2017|Councilmember|Adam Medrano       |
|02/08/2017|Councilmember|Casey Thomas       |
|02/08/2017|Councilmember|Carolyn King Arnold|
|02/08/2017|Councilmember|Scott Griggs       |
|02/08/2017|Councilmember|B. Adam  McGough   |
|02/08/2017|Councilmember|Lee Kleinman       |
|02/08/2017|Councilmember|Sandy Greyson      |
+----------+-------------+-------------------+
only showing top 10 rows


### 6.2. Lọc dữ liệu

In [15]:
#voter_df.select("VOTER_NAME").distinct().show(20, truncate=False)

voter_clean = (
    voter_df
    .filter(F.col("VOTER_NAME").isNotNull())
    .filter(F.length("VOTER_NAME").between(1, 20))
    .filter(~F.col("VOTER_NAME").contains("_"))
)

#voter_clean.select("VOTER_NAME").distinct().show(20, truncate=False)

### 6.3. Biến đổi chuỗi

Các hàm thường dùng trong `pyspark.sql.functions`:

- `upper()`, `lower()`
- `trim()`
- `split()`
- `length()`
- `contains()`
- `regexp_replace()`

In [16]:
voter_string = (
    voter_df
    .filter(F.col("VOTER_NAME").isNotNull())
    .withColumn("NAME_UPPER", F.upper("VOTER_NAME"))
    .withColumn("NAME_PARTS", F.split("VOTER_NAME", " "))
    .withColumn("PART_COUNT", F.size("NAME_PARTS"))
)

voter_string.select(
    "VOTER_NAME", "NAME_UPPER", "NAME_PARTS", "PART_COUNT"
).show(10, truncate=False)

+-------------------+-------------------+-----------------------+----------+
|VOTER_NAME         |NAME_UPPER         |NAME_PARTS             |PART_COUNT|
+-------------------+-------------------+-----------------------+----------+
|Jennifer S. Gates  |JENNIFER S. GATES  |[Jennifer, S., Gates]  |3         |
|Philip T. Kingston |PHILIP T. KINGSTON |[Philip, T., Kingston] |3         |
|Michael S. Rawlings|MICHAEL S. RAWLINGS|[Michael, S., Rawlings]|3         |
|Adam Medrano       |ADAM MEDRANO       |[Adam, Medrano]        |2         |
|Casey Thomas       |CASEY THOMAS       |[Casey, Thomas]        |2         |
|Carolyn King Arnold|CAROLYN KING ARNOLD|[Carolyn, King, Arnold]|3         |
|Scott Griggs       |SCOTT GRIGGS       |[Scott, Griggs]        |2         |
|B. Adam  McGough   |B. ADAM  MCGOUGH   |[B., Adam, , McGough]  |4         |
|Lee Kleinman       |LEE KLEINMAN       |[Lee, Kleinman]        |2         |
|Sandy Greyson      |SANDY GREYSON      |[Sandy, Greyson]       |2         |

### 6.4. Tách first name / last name

In [17]:
voter_names = (
    voter_df
    .filter(F.col("VOTER_NAME").isNotNull())
    .withColumn("name_parts", F.split(F.trim("VOTER_NAME"), " "))
    .withColumn("first_name", F.col("name_parts").getItem(0))
    .withColumn(
        "last_name",
        F.col("name_parts").getItem(
            F.size("name_parts") - 1
        )
    )
)

voter_names.select(
    "VOTER_NAME", "first_name", "last_name"
).show(10, truncate=False)

/usr/local/lib/python3.13/dist-packages/pyspark/sql/classic/column.py:359: FutureWarning: A column as 'key' in getItem is deprecated as of Spark 3.0, and will not be supported in the future release. Use `column[key]` or `column.key` syntax instead.
  warnings.warn(


+-------------------+----------+---------+
|VOTER_NAME         |first_name|last_name|
+-------------------+----------+---------+
|Jennifer S. Gates  |Jennifer  |Gates    |
|Philip T. Kingston |Philip    |Kingston |
|Michael S. Rawlings|Michael   |Rawlings |
|Adam Medrano       |Adam      |Medrano  |
|Casey Thomas       |Casey     |Thomas   |
|Carolyn King Arnold|Carolyn   |Arnold   |
|Scott Griggs       |Scott     |Griggs   |
|B. Adam  McGough   |B.        |McGough  |
|Lee Kleinman       |Lee       |Kleinman |
|Sandy Greyson      |Sandy     |Greyson  |
+-------------------+----------+---------+
only showing top 10 rows


## Bài 7. Conditional DataFrame Column Operations

Spark hỗ trợ logic tương tự `if / elif / else` qua:

- `F.when()`
- `.otherwise()`

In [18]:
age_df = spark.createDataFrame([
    ("Alice", 14),
    ("Bob", 18),
    ("Candice", 38)
], ["Name", "Age"])

classified_df = age_df.withColumn(
    "Age_Group",
    F.when(F.col("Age") >= 18, "Adult")
     .otherwise("Minor")
)

classified_df.show()

+-------+---+---------+
|   Name|Age|Age_Group|
+-------+---+---------+
|  Alice| 14|    Minor|
|    Bob| 18|    Adult|
|Candice| 38|    Adult|
+-------+---+---------+



### 7.1. Nhiều nhánh điều kiện

In [19]:
multi_df = age_df.withColumn(
    "Category",
    F.when(F.col("Age") < 13, "Child")
     .when(F.col("Age") < 18, "Teen")
     .otherwise("Adult")
)

multi_df.show()

+-------+---+--------+
|   Name|Age|Category|
+-------+---+--------+
|  Alice| 14|    Teen|
|    Bob| 18|   Adult|
|Candice| 38|   Adult|
+-------+---+--------+



## Bài 8. User Defined Functions - UDF

UDF cho phép dùng hàm Python tùy biến trong DataFrame.

Quy trình:

1. viết hàm Python;
2. bọc hàm bằng `F.udf`;
3. khai báo kiểu dữ liệu trả về;
4. dùng như một hàm Spark.

> Trong thực tế, nếu có thể biểu diễn bằng built-in Spark functions thì nên ưu tiên built-in functions vì thường tối ưu hơn Python UDF.

### 8.1. UDF đảo chuỗi

In [20]:
def reverse_string(text):
    if text is None:
        return None
    return text[::-1]

udf_reverse = F.udf(reverse_string, StringType())


voter_udf = voter_df.withColumn(
    "ReverseName",
    udf_reverse("VOTER_NAME")
)

voter_udf.select("VOTER_NAME", "ReverseName").show(10, truncate=False)

+-------------------+-------------------+
|VOTER_NAME         |ReverseName        |
+-------------------+-------------------+
|Jennifer S. Gates  |setaG .S refinneJ  |
|Philip T. Kingston |notsgniK .T pilihP |
|Michael S. Rawlings|sgnilwaR .S leahciM|
|Adam Medrano       |onardeM madA       |
|Casey Thomas       |samohT yesaC       |
|Carolyn King Arnold|dlonrA gniK nyloraC|
|Scott Griggs       |sggirG ttocS       |
|B. Adam  McGough   |hguoGcM  madA .B   |
|Lee Kleinman       |namnielK eeL       |
|Sandy Greyson      |nosyerG ydnaS      |
+-------------------+-------------------+
only showing top 10 rows


## 8.2. UDF đếm số từ trong tên

In [21]:
def name_word_count(name):
    if name is None:
        return 0
    return len(name.split())

udf_name_word_count = F.udf(name_word_count, IntegerType())


voter_udf = voter_udf.withColumn(
    "NameWordCount",
    udf_name_word_count("VOTER_NAME")
)

voter_udf.select(
    "VOTER_NAME", "NameWordCount"
).show(10, truncate=False)

+-------------------+-------------+
|VOTER_NAME         |NameWordCount|
+-------------------+-------------+
|Jennifer S. Gates  |3            |
|Philip T. Kingston |3            |
|Michael S. Rawlings|3            |
|Adam Medrano       |2            |
|Casey Thomas       |2            |
|Carolyn King Arnold|3            |
|Scott Griggs       |2            |
|B. Adam  McGough   |3            |
|Lee Kleinman       |2            |
|Sandy Greyson      |2            |
+-------------------+-------------+
only showing top 10 rows


<a id="partition"></a>
# Bài 9. Partitioning và ID

Spark chia DataFrame thành nhiều **partition**.  
Mỗi partition có thể được xử lý độc lập trên các worker.

Điều này giúp xử lý song song, nhưng cũng khiến việc tạo ID tuần tự `0,1,2,...` không tự nhiên trong môi trường phân tán.

## 9.1. Xem số partition

In [22]:
print("Partitions:", voter_df.rdd.getNumPartitions())

Partitions: 1


## 9.2. `monotonically_increasing_id()`

Hàm này tạo ID:

- 64-bit;
- duy nhất;
- tăng dần về mặt giá trị;
- **không đảm bảo liên tục**.

In [23]:
voter_with_id = voter_df.withColumn(
    "ID",
    F.monotonically_increasing_id()
)

voter_with_id.select("ID", "VOTER_NAME").show(10, truncate=False)

+---+-------------------+
|ID |VOTER_NAME         |
+---+-------------------+
|0  |Jennifer S. Gates  |
|1  |Philip T. Kingston |
|2  |Michael S. Rawlings|
|3  |Adam Medrano       |
|4  |Casey Thomas       |
|5  |Carolyn King Arnold|
|6  |Scott Griggs       |
|7  |B. Adam  McGough   |
|8  |Lee Kleinman       |
|9  |Sandy Greyson      |
+---+-------------------+
only showing top 10 rows


## Bài 10. Caching

Caching lưu DataFrame ở memory/disk để tái sử dụng nhanh hơn.

Phù hợp khi:

- một DataFrame được dùng nhiều lần;
- các bước trước đó tốn nhiều chi phí;
- dữ liệu đủ phù hợp với tài nguyên hệ thống.

Không nên cache mọi thứ.

In [24]:
cached_df = voter_df.cache()

# Materialize cache
row_count = cached_df.count()

print("Rows:", row_count)
print("Is cached:", cached_df.is_cached)

cached_df.unpersist()
print("Is cached:", cached_df.is_cached)

Rows: 44625
Is cached: True
Is cached: False


## Bài 11. Tối ưu nhập dữ liệu

Một số nguyên tắc:

- định nghĩa schema rõ ràng;
- dữ liệu nên được chia thành các file có kích thước tương đối cân bằng;
- có thể đọc nhiều file bằng wildcard;
- dữ liệu trung gian nên chuyển sang Parquet nếu phù hợp.

In [25]:
wildcard_path = str(DATA_DIR / "AA_DFW_*_Departures_Short.csv.gz")

flights_wildcard = spark.read.csv(
    wildcard_path,
    header=True,
    schema=flight_schema
)

print("Số dòng:", flights_wildcard.count())

Số dòng: 583718


## Bài 12. Execution Plan, Shuffle và Broadcast

### 12.1. Execution Plan

`explain()` giúp xem Spark sẽ thực hiện truy vấn như thế nào.

In [26]:
voter_df.select("VOTER_NAME").distinct().explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[VOTER_NAME#205], functions=[])
   +- Exchange hashpartitioning(VOTER_NAME#205, 200), ENSURE_REQUIREMENTS, [plan_id=525]
      +- HashAggregate(keys=[VOTER_NAME#205], functions=[])
         +- FileScan csv [VOTER_NAME#205] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/Datasets/DallasCouncilVoters.csv.gz], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<VOTER_NAME:string>




### 12.2. Shuffle

Shuffle xảy ra khi Spark phải di chuyển dữ liệu giữa các partition/worker.

Các thao tác dễ phát sinh shuffle:

- `groupBy`
- `distinct`
- `join`
- `repartition`

Shuffle thường tốn network và I/O.

### 12.3. `coalesce()` và `repartition()`

- `repartition(n)`: có thể tăng/giảm partition, thường gây shuffle.
- `coalesce(n)`: chủ yếu dùng để giảm số partition với ít shuffle hơn.

In [27]:
print("Ban đầu:", flights_wildcard.rdd.getNumPartitions())

repart_df = flights_wildcard.repartition(4)
print("repartition(4):", repart_df.rdd.getNumPartitions())

coalesce_df = repart_df.coalesce(2)
print("coalesce(2):", coalesce_df.rdd.getNumPartitions())

Ban đầu: 2
repartition(4): 4
coalesce(2): 2


## 12.4. Broadcast Join

Nếu một DataFrame nhỏ, Spark có thể gửi bản sao tới các worker để giảm shuffle.

In [28]:
airport_info = spark.createDataFrame([
    ("HNL", "Honolulu"),
    ("OGG", "Kahului"),
    ("SFO", "San Francisco"),
    ("LAS", "Las Vegas"),
    ("DTW", "Detroit")
], ["airport_code", "airport_name"])


joined_broadcast = flights_wildcard.join(
    F.broadcast(airport_info),
    flights_wildcard["Destination Airport"] == airport_info["airport_code"],
    "left"
)

joined_broadcast.select(
    "Destination Airport", "airport_name"
).show(10)

+-------------------+------------+
|Destination Airport|airport_name|
+-------------------+------------+
|                HNL|    Honolulu|
|                OGG|     Kahului|
|                SLC|        NULL|
|                DTW|     Detroit|
|                PIT|        NULL|
|                SAN|        NULL|
|                MIA|        NULL|
|                JFK|        NULL|
|                ORD|        NULL|
|                STL|        NULL|
+-------------------+------------+
only showing top 10 rows


## 13. Data Pipeline

Một data pipeline gồm:

```text
Input
  ↓
Validation
  ↓
Cleaning
  ↓
Transformation
  ↓
Analysis
  ↓
Output
```

Nguồn có thể là:

- CSV
- JSON
- Parquet
- Database
- Web service

Output có thể là:

- Parquet
- CSV
- Database
- Dashboard / BI

### 13.1. Pipeline mẫu cho Dallas Council Votes

Ta sẽ:

1. đọc dữ liệu;
2. chuẩn hóa tên cột;
3. loại voter null;
4. chuẩn hóa chữ;
5. thêm ID;
6. tạo bảng phân tích.

In [29]:
votes_df = spark.read.csv(
    str(DATA_DIR / "DallasCouncilVotes.csv.gz"),
    header=True,
    inferSchema=True
)

votes_df.printSchema()

votes_clean = (
    votes_df
    .withColumnRenamed("VOTER NAME", "VOTER_NAME")
    .withColumnRenamed("VOTE CAST", "VOTE_CAST")
    .filter(F.col("VOTER_NAME").isNotNull())
    .withColumn("VOTER_NAME", F.trim(F.col("VOTER_NAME")))
    .withColumn("VOTER_NAME_UPPER", F.upper("VOTER_NAME"))
    .withColumn("ROW_ID", F.monotonically_increasing_id())
)

votes_clean.groupBy("VOTE_CAST").count().show()

root
 |-- DATE: string (nullable = true)
 |-- AGENDA_ITEM_NUMBER: string (nullable = true)
 |-- ITEM_TYPE: string (nullable = true)
 |-- DISTRICT: string (nullable = true)
 |-- TITLE: string (nullable = true)
 |-- VOTER NAME: string (nullable = true)
 |-- VOTE CAST: string (nullable = true)
 |-- FINAL ACTION TAKEN: string (nullable = true)
 |-- AGENDA ITEM DESCRIPTION: string (nullable = true)
 |-- AGENDA_ID: string (nullable = true)
 |-- VOTE_ID: string (nullable = true)

+--------------------+-----+
|           VOTE_CAST|count|
+--------------------+-----+
|  for  services  ...|   15|
|       011018__42_10|    1|
| for the services...|   15|
|                AWVT| 1116|
|       011018__42_11|    1|
|  for  services  ...|   15|
|        011018__42_8|    1|
|  for  services  ...|   15|
|                 YES|39983|
|  for  services  ...|   15|
|               ABSNT|  642|
|        011018__42_2|    1|
|  for  services  ...|   15|
|       011018__42_15|    1|
| for services and...|   15|


### 13.2. Xuất dữ liệu sạch

In [30]:
votes_clean.write.mode("overwrite").parquet(
    "DallasCouncilVotes_Clean.parquet"
    )

## 14. Data Validation

Dữ liệu CSV thực tế có thể chứa:

- dòng trống;
- comment;
- header;
- số cột không đồng nhất;
- nhiều delimiter;
- cấu trúc lồng nhau;
- dữ liệu sai kiểu.

Spark hỗ trợ nhiều tùy chọn khi đọc CSV.

**Validation** là kiểm tra dữ liệu có phù hợp với format/rule mong đợi không.

Có thể kiểm tra:

- số dòng;
- số cột;
- kiểu dữ liệu;
- miền giá trị;
- khóa hợp lệ;
- quan hệ với bảng tham chiếu.

### 14.1. Validation bằng join

In [31]:
valid_titles = spark.createDataFrame([
    ("Mayor",),
    ("Councilmember",)
], ["TITLE"])


before = voter_df.count()

verified_voters = voter_df.join(
    F.broadcast(valid_titles),
    on="TITLE",
    how="inner"
)

after = verified_voters.count()

print("Before:", before)
print("After :", after)

Before: 44625
After : 38020


### 14.2. Tìm dữ liệu không hợp lệ bằng left_anti

In [32]:
invalid_voters = voter_df.join(
  F.broadcast(valid_titles),
  on="TITLE",
  how="left_anti"
  )

invalid_voters.show(10, truncate=False)

+--------------------+----------+-----------------+
|TITLE               |DATE      |VOTER_NAME       |
+--------------------+----------+-----------------+
|Deputy Mayor Pro Tem|04/25/2018|Adam Medrano     |
|Mayor Pro Tem       |04/25/2018|Dwaine R. Caraway|
|Deputy Mayor Pro Tem|06/20/2018|Adam Medrano     |
|Mayor Pro Tem       |06/20/2018|Dwaine R. Caraway|
|Deputy Mayor Pro Tem|06/20/2018|Adam Medrano     |
|Mayor Pro Tem       |06/20/2018|Dwaine R. Caraway|
|Deputy Mayor Pro Tem|08/15/2018|Adam Medrano     |
|Vacant              |08/15/2018|NULL             |
|Deputy Mayor Pro Tem|08/15/2018|Adam Medrano     |
|Vacant              |08/15/2018|NULL             |
+--------------------+----------+-----------------+
only showing top 10 rows


## BÀI TẬP MỞ RỘNG

Mục tiêu: xây dựng một pipeline hoàn chỉnh trên 4 tệp chuyến bay 2014-2017.

**Yêu cầu**

Từ 4 file chuyến bay 2014-2017:

1. đọc bằng schema cố định;
2. gộp dữ liệu;
3. chuẩn hóa tên cột;
4. loại thời gian bay null hoặc <= 0;
5. thêm cột `year`;
6. chuẩn hóa airport thành uppercase;
7. tính thống kê theo năm và destination;
8. lưu kết quả thành Parquet.

In [33]:
# Câu 1–2: Đọc dữ liệu bằng schema cố định và gộp 4 file chuyến bay 2014–2017
print("Câu 1–2: Đọc dữ liệu từ 4 file chuyến bay năm 2014–2017 bằng schema cố định và gộp thành một DataFrame.")

flight_schema = StructType([
    StructField("Date (MM/DD/YYYY)", StringType(), True),
    StructField("Flight Number", StringType(), True),
    StructField("Destination Airport", StringType(), True),
    StructField("Actual elapsed time (Minutes)", IntegerType(), True)
])

wildcard_path = str(DATA_DIR / "AA_DFW_*_Departures_Short.csv.gz")

flights_wildcard = spark.read.csv(
    wildcard_path,
    header=True,
    schema=flight_schema
)

print("Số dòng:", flights_wildcard.count())


# Câu 3–6: Chuẩn hóa tên cột, làm sạch dữ liệu, tạo cột year và chuẩn hóa sân bay đến
print("Câu 3–6: Chuẩn hóa tên cột, loại bỏ thời gian bay null hoặc <= 0, tạo cột year và chuyển mã sân bay đến thành chữ in hoa.")

clean_flights = (
    flights_wildcard
    .withColumnRenamed("Date (MM/DD/YYYY)", "flight_date")
    .withColumnRenamed("Flight Number", "flight_number")
    .withColumnRenamed("Destination Airport", "destination")
    .withColumnRenamed(
        "Actual elapsed time (Minutes)",
        "elapsed_minutes"
    )
    .filter(F.col("elapsed_minutes").isNotNull())
    .filter(F.col("elapsed_minutes") > 0)
    .withColumn(
        "flight_date_parsed",
        F.to_date("flight_date", "MM/dd/yyyy")
    )
    .filter(F.col("flight_date_parsed").isNotNull())
    .withColumn("year", F.year("flight_date_parsed"))
    .withColumn("destination", F.upper(F.trim(F.col("destination"))))
)

clean_flights.printSchema()
clean_flights.show(10, truncate=False)


# Câu 7: Tính thống kê theo năm và sân bay đến
print("Câu 7: Tính số chuyến bay, thời gian bay trung bình, nhỏ nhất và lớn nhất theo năm và sân bay đến.")

flight_stats = (
    clean_flights
    .groupBy("year", "destination")
    .agg(
        F.count("*").alias("num_flights"),
        F.round(F.avg("elapsed_minutes"), 2).alias("avg_minutes"),
        F.min("elapsed_minutes").alias("min_minutes"),
        F.max("elapsed_minutes").alias("max_minutes")
    )
    .orderBy("year", F.desc("num_flights"))
)

flight_stats.show(30, truncate=False)


# Câu 8: Lưu kết quả thành Parquet
print("Câu 8: Lưu bảng thống kê chuyến bay thành định dạng Parquet.")

PROJECT_OUTPUT = str(Path.cwd() / "flight_clean_stats.parquet")

(
    flight_stats.write
    .mode("overwrite")
    .parquet(PROJECT_OUTPUT)
)

print("Đã ghi:", PROJECT_OUTPUT)

Câu 1–2: Đọc dữ liệu từ 4 file chuyến bay năm 2014–2017 bằng schema cố định và gộp thành một DataFrame.
Số dòng: 583718
Câu 3–6: Chuẩn hóa tên cột, loại bỏ thời gian bay null hoặc <= 0, tạo cột year và chuyển mã sân bay đến thành chữ in hoa.
root
 |-- flight_date: string (nullable = true)
 |-- flight_number: string (nullable = true)
 |-- destination: string (nullable = true)
 |-- elapsed_minutes: integer (nullable = true)
 |-- flight_date_parsed: date (nullable = true)
 |-- year: integer (nullable = true)

+-----------+-------------+-----------+---------------+------------------+----+
|flight_date|flight_number|destination|elapsed_minutes|flight_date_parsed|year|
+-----------+-------------+-----------+---------------+------------------+----+
|01/01/2014 |0005         |HNL        |519            |2014-01-01        |2014|
|01/01/2014 |0007         |OGG        |505            |2014-01-01        |2014|
|01/01/2014 |0035         |SLC        |174            |2014-01-01        |2014|
|01/01/2

In [34]:
PROJECT_OUTPUT = str(Path.cwd() / "flight_clean_stats.parquet")

(
    flight_stats.write
    .mode("overwrite")
    .parquet(PROJECT_OUTPUT)
)

print("Đã ghi:", PROJECT_OUTPUT)

Đã ghi: /content/flight_clean_stats.parquet


**Câu hỏi phân tích**

* Destination nào có nhiều chuyến bay nhất trong từng năm?
* Destination nào có thời gian bay trung bình lớn nhất?
* Có bản ghi nào có elapsed_minutes bất thường không?
* Nếu chuyển pipeline sang Parquet, những bước nào có thể chạy hiệu quả hơn?
* Có nên cache clean_flights trước khi chạy nhiều phép phân tích khác nhau? Giải thích.

In [35]:
from time import perf_counter
from pyspark.sql.window import Window


# Câu 8: Destination nào có nhiều chuyến bay nhất trong từng năm?
print("\nCâu 8: Destination nào có nhiều chuyến bay nhất trong từng năm?")

w = Window.partitionBy("year").orderBy(F.desc("num_flights"))

(
    flight_stats
    .withColumn("rank", F.rank().over(w))
    .filter(F.col("rank") == 1)
    .select("year", "destination", "num_flights")
    .orderBy("year", "destination")
    .show(truncate=False)
)


# Câu 9: Destination nào có thời gian bay trung bình lớn nhất?
print("\nCâu 9: Destination nào có thời gian bay trung bình lớn nhất?")

(
    flight_stats
    .orderBy(F.desc("avg_minutes"))
    .select("year", "destination", "avg_minutes", "num_flights")
    .show(10, truncate=False)
)


# Câu 10: Có bản ghi nào có elapsed_minutes bất thường không?
print("\nCâu 10: Có bản ghi nào có elapsed_minutes bất thường không?")

q1, q3 = clean_flights.approxQuantile(
    "elapsed_minutes", [0.25, 0.75], 0.01
)

iqr = q3 - q1
lower_bound = max(0, q1 - 1.5 * iqr)
upper_bound = q3 + 1.5 * iqr

print(f"Q1 = {q1}, Q3 = {q3}, IQR = {iqr}")
print(f"Ngưỡng dưới = {lower_bound}, ngưỡng trên = {upper_bound}")

outliers = clean_flights.filter(
    (F.col("elapsed_minutes") < lower_bound) |
    (F.col("elapsed_minutes") > upper_bound)
)

print("Số bản ghi ngoại lệ tiềm năng:", outliers.count())

outliers.select(
    "flight_date", "flight_number", "destination", "elapsed_minutes"
).orderBy(
    F.desc("elapsed_minutes")
).show(20, truncate=False)


# Câu 11: Nếu chuyển pipeline sang Parquet, những bước nào có thể chạy hiệu quả hơn?
print("\nCâu 11: Nếu chuyển pipeline sang Parquet, những bước nào có thể chạy hiệu quả hơn?")

parquet_flights = spark.read.parquet(PROJECT_OUTPUT)

print("Schema của dữ liệu Parquet:")
parquet_flights.printSchema()

print("Số dòng trong Parquet:", parquet_flights.count())

print("""
Nhận xét:
- Parquet lưu dữ liệu theo cột và hỗ trợ nén.
- Đọc chọn lọc cột và lọc dữ liệu có thể giảm lượng I/O.
- Các lần phân tích sau có thể tránh đọc và xử lý lại CSV.
- Việc tăng tốc thực tế cần được xác nhận bằng benchmark.
""")


# Câu 12: Có nên cache clean_flights trước khi chạy nhiều phép phân tích khác nhau?
print("\nCâu 12: Có nên cache clean_flights trước khi chạy nhiều phép phân tích khác nhau?")

print("Execution plan của clean_flights:")
clean_flights.explain()

print("""
Nhận xét:
- Nên cân nhắc cache nếu clean_flights được sử dụng lặp lại
  trong nhiều phép phân tích.
- Cache có thể tránh tính toán lại các bước đọc, chuẩn hóa và lọc.
- Cache tiêu tốn bộ nhớ; không nhất thiết có lợi nếu chỉ chạy một lần.
- Nếu cache, hãy materialize dữ liệu bằng một action và unpersist
  khi hoàn tất để giải phóng tài nguyên.
""")



Câu 8: Destination nào có nhiều chuyến bay nhất trong từng năm?
+----+-----------+-----------+
|year|destination|num_flights|
+----+-----------+-----------+
|2014|LAX        |5990       |
|2015|LAX        |5934       |
|2016|LAX        |5392       |
|2017|LAX        |4705       |
+----+-----------+-----------+


Câu 9: Destination nào có thời gian bay trung bình lớn nhất?
+----+-----------+-----------+-----------+
|year|destination|avg_minutes|num_flights|
+----+-----------+-----------+-----------+
|2017|LIH        |536.47     |17         |
|2016|HNL        |511.23     |744        |
|2016|KOA        |509.65     |17         |
|2015|HNL        |508.14     |749        |
|2017|HNL        |507.81     |687        |
|2016|OGG        |500.63     |519        |
|2014|HNL        |500.42     |723        |
|2017|OGG        |499.19     |531        |
|2015|OGG        |496.55     |533        |
|2017|KOA        |496.52     |128        |
+----+-----------+-----------+-----------+
only showing top 10 ro

## 4. Bài tập về nhà

### Bài 1 - Schema

Tạo schema cho dữ liệu:

| product_id | product_name | price | quantity |
|---|---|---:|---:|

Yêu cầu:

- `product_id`: Integer
- `product_name`: String
- `price`: Double
- `quantity`: Integer

---

### Bài 2 - Cleaning

Với `DallasCouncilVoters.csv.gz`:

1. loại `VOTER_NAME` null;
2. trim khoảng trắng;
3. đổi tên thành uppercase;
4. tách tên thành array;
5. đếm số thành phần của tên.

---

### Bài 3 - Conditional Column

Tạo cột `flight_type`:

- `< 120 phút`: `Short`
- `120-240 phút`: `Medium`
- `> 240 phút`: `Long`

---

### Bài 4 - Parquet

1. đọc 4 file chuyến bay;
2. union;
3. ghi Parquet;
4. đọc lại;
5. so sánh schema và số dòng.

---

### Bài 5 - Cache

Thực hiện cùng một phép `groupBy()`:

1. trước khi cache;
2. sau khi cache;
3. `unpersist()` sau khi dùng.

Quan sát thời gian chạy.

---

### Bài 6 - Broadcast Join

Tạo bảng tra cứu mã airport nhỏ và join với dữ liệu chuyến bay.

So sánh:

```python
df.join(lookup_df, ...)
```

và

```python
df.join(F.broadcast(lookup_df), ...)
```

Dùng `explain()` để quan sát kế hoạch.

---

### Bài 7 - Validation

Với Dallas Council Votes:

1. xác định danh sách `VOTE CAST`;
2. tạo danh sách giá trị hợp lệ;
3. dùng join để giữ valid rows;
4. dùng `left_anti` để lấy invalid rows.

### Giải Bài 1 - Schema
Tạo schema tường minh cho cấu trúc bảng sản phẩm và khởi tạo DataFrame minh họa.

In [36]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

# 1. Định nghĩa schema
product_schema = StructType([
    StructField("product_id", IntegerType(), True),
    StructField("product_name", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("quantity", IntegerType(), True)
])

# 2. Tạo dữ liệu mẫu và áp dụng schema
sample_products = [
    (101, "Laptop", 1200.50, 5),
    (102, "Mouse", 25.00, 15),
    (103, "Keyboard", 75.20, None)
]

product_df = spark.createDataFrame(sample_products, schema=product_schema)
product_df.printSchema()
product_df.show()

root
 |-- product_id: integer (nullable = true)
 |-- product_name: string (nullable = true)
 |-- price: double (nullable = true)
 |-- quantity: integer (nullable = true)

+----------+------------+------+--------+
|product_id|product_name| price|quantity|
+----------+------------+------+--------+
|       101|      Laptop|1200.5|       5|
|       102|       Mouse|  25.0|      15|
|       103|    Keyboard|  75.2|    NULL|
+----------+------------+------+--------+



### Giải Bài 2 - Cleaning
Thực hiện các bước làm sạch trên tập dữ liệu cử tri Dallas.

In [37]:
voter_clean_ex = (
    voter_df
    # 1. Loại VOTER_NAME null
    .filter(F.col("VOTER_NAME").isNotNull())
    # 2. Trim khoảng trắng
    .withColumn("VOTER_NAME", F.trim(F.col("VOTER_NAME")))
    # 3. Đổi tên thành uppercase
    .withColumn("VOTER_NAME_UPPER", F.upper(F.col("VOTER_NAME")))
    # 4. Tách tên thành array
    .withColumn("NAME_ARRAY", F.split(F.col("VOTER_NAME_UPPER"), " "))
    # 5. Đếm số thành phần của tên
    .withColumn("NAME_PART_COUNT", F.size(F.col("NAME_ARRAY")))
)

voter_clean_ex.select("VOTER_NAME", "VOTER_NAME_UPPER", "NAME_ARRAY", "NAME_PART_COUNT").show(10, truncate=False)

+-------------------+-------------------+-----------------------+---------------+
|VOTER_NAME         |VOTER_NAME_UPPER   |NAME_ARRAY             |NAME_PART_COUNT|
+-------------------+-------------------+-----------------------+---------------+
|Jennifer S. Gates  |JENNIFER S. GATES  |[JENNIFER, S., GATES]  |3              |
|Philip T. Kingston |PHILIP T. KINGSTON |[PHILIP, T., KINGSTON] |3              |
|Michael S. Rawlings|MICHAEL S. RAWLINGS|[MICHAEL, S., RAWLINGS]|3              |
|Adam Medrano       |ADAM MEDRANO       |[ADAM, MEDRANO]        |2              |
|Casey Thomas       |CASEY THOMAS       |[CASEY, THOMAS]        |2              |
|Carolyn King Arnold|CAROLYN KING ARNOLD|[CAROLYN, KING, ARNOLD]|3              |
|Scott Griggs       |SCOTT GRIGGS       |[SCOTT, GRIGGS]        |2              |
|B. Adam  McGough   |B. ADAM  MCGOUGH   |[B., ADAM, , MCGOUGH]  |4              |
|Lee Kleinman       |LEE KLEINMAN       |[LEE, KLEINMAN]        |2              |
|Sandy Greyson  

### Giải Bài 3 - Conditional Column
Tạo cột phân loại `flight_type` dựa trên thời gian bay.

In [38]:
flight_type_df = clean_flights.withColumn(
    "flight_type",
    F.when(F.col("elapsed_minutes") < 120, "Short")
     .when(F.col("elapsed_minutes") <= 240, "Medium")
     .otherwise("Long")
)

flight_type_df.groupBy("flight_type").count().show()

+-----------+------+
|flight_type| count|
+-----------+------+
|     Medium|382156|
|       Long| 25823|
|      Short|166017|
+-----------+------+



### Giải Bài 4 - Parquet
Gộp 4 file chuyến bay, lưu trữ dưới dạng Parquet, đọc lại và đối chiếu kết quả.

In [39]:
# 1 & 2. Đọc và union 4 file
files = [
    "AA_DFW_2014_Departures_Short.csv.gz",
    "AA_DFW_2015_Departures_Short.csv.gz",
    "AA_DFW_2016_Departures_Short.csv.gz",
    "AA_DFW_2017_Departures_Short.csv.gz"
]
flight_dfs = [
    spark.read.csv(str(DATA_DIR / f), header=True, schema=flight_schema)
    for f in files
]
flights_union = flight_dfs[0]
for df in flight_dfs[1:]:
    flights_union = flights_union.unionByName(df)

# 3. Ghi Parquet
PARQUET_EX_PATH = "AA_DFW_EX_ALL.parquet"
flights_union.write.mode("overwrite").parquet(PARQUET_EX_PATH)

# 4. Đọc lại
flights_parquet_ex = spark.read.parquet(PARQUET_EX_PATH)

# 5. So sánh
print("--- Gốc (Union) ---")
print("Số dòng:", flights_union.count())
flights_union.printSchema()

print("--- Đọc lại từ Parquet ---")
print("Số dòng:", flights_parquet_ex.count())
flights_parquet_ex.printSchema()

--- Gốc (Union) ---
Số dòng: 583718
root
 |-- Date (MM/DD/YYYY): string (nullable = true)
 |-- Flight Number: string (nullable = true)
 |-- Destination Airport: string (nullable = true)
 |-- Actual elapsed time (Minutes): integer (nullable = true)

--- Đọc lại từ Parquet ---
Số dòng: 583718
root
 |-- Date (MM/DD/YYYY): string (nullable = true)
 |-- Flight Number: string (nullable = true)
 |-- Destination Airport: string (nullable = true)
 |-- Actual elapsed time (Minutes): integer (nullable = true)



### Giải Bài 5 - Cache
So sánh hiệu năng truy vấn `groupBy` trước và sau khi kích hoạt Cache.

In [40]:
import time

# Đảm bảo unpersist trước khi đo
voter_df.unpersist()

# 1. Trước khi cache
start_time = time.perf_counter()
count_before = voter_df.groupBy("TITLE").count().collect()
end_time = time.perf_counter()
time_before = end_time - start_time
print(f"Thời gian chạy trước khi cache: {time_before:.4f} giây")

# 2. Sau khi cache
voter_df.cache()
# Thực hiện hành động (Action) đầu tiên để khởi tạo vùng nhớ đệm (Materialize cache)
voter_df.count()

start_time = time.perf_counter()
count_after = voter_df.groupBy("TITLE").count().collect()
end_time = time.perf_counter()
time_after = end_time - start_time
print(f"Thời gian chạy sau khi đã cache: {time_after:.4f} giây")

# 3. Giải phóng bộ nhớ
voter_df.unpersist()

Thời gian chạy trước khi cache: 0.7972 giây
Thời gian chạy sau khi đã cache: 0.2387 giây


DataFrame[DATE: string, TITLE: string, VOTER_NAME: string]

### Giải Bài 6 - Broadcast Join
So sánh kế hoạch thực thi giữa phép Join thông thường và Broadcast Join.

In [41]:
# Tạo bảng tra cứu nhỏ
lookup_df = spark.createDataFrame([
    ("HNL", "Honolulu"),
    ("OGG", "Kahului"),
    ("SLC", "Salt Lake City")
], ["airport_code", "airport_name"])

print("=== KẾ HOẠCH thực thi của JOIN thông thường ===")
normal_join = clean_flights.join(lookup_df, clean_flights["destination"] == lookup_df["airport_code"])
normal_join.explain()

print("\n=== KẾ HOẠCH thực thi của BROADCAST JOIN ===")
broadcast_join = clean_flights.join(F.broadcast(lookup_df), clean_flights["destination"] == lookup_df["airport_code"])
broadcast_join.explain()

=== KẾ HOẠCH thực thi của JOIN thông thường ===
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- BroadcastHashJoin [destination#563], [airport_code#1098], Inner, BuildLeft, false
   :- BroadcastExchange HashedRelationBroadcastMode(List(input[2, string, true]),false), [plan_id=2137]
   :  +- Project [flight_date#557, flight_number#558, upper(trim(destination#559, None)) AS destination#563, elapsed_minutes#560, flight_date_parsed#561, year(flight_date_parsed#561) AS year#562]
   :     +- Project [Date (MM/DD/YYYY)#544 AS flight_date#557, Flight Number#545 AS flight_number#558, Destination Airport#546 AS destination#559, Actual elapsed time (Minutes)#547 AS elapsed_minutes#560, cast(gettimestamp(Date (MM/DD/YYYY)#544, MM/dd/yyyy, TimestampType, try_to_timestamp, Some(Etc/UTC), true) as date) AS flight_date_parsed#561]
   :        +- Filter (((isnotnull(Actual elapsed time (Minutes)#547) AND (Actual elapsed time (Minutes)#547 > 0)) AND isnotnull(cast(gettimestamp(Date (MM/DD/YYYY

### Giải Bài 7 - Validation
Xác thực dữ liệu phiếu bầu bằng cách dùng inner join để lọc dòng hợp lệ và left_anti join để tìm các dòng không hợp lệ.

In [42]:
# 1 & 2. Xác định và tạo danh sách VOTE_CAST hợp lệ mẫu
valid_votes = spark.createDataFrame([
    ("YES",),
    ("AWVT",),
    ("ABSNT",),
    ("N/A",)
], ["VOTE_CAST"])

# 3. Dùng join để giữ lại valid rows
valid_rows = votes_clean.join(
    F.broadcast(valid_votes),
    on="VOTE_CAST",
    how="inner"
)
print("Số dòng hợp lệ:", valid_rows.count())

# 4. Dùng left_anti để lấy invalid rows
invalid_rows = votes_clean.join(
    F.broadcast(valid_votes),
    on="VOTE_CAST",
    how="left_anti"
)
print("Số dòng không hợp lệ:", invalid_rows.count())
invalid_rows.select("ROW_ID", "VOTER_NAME", "VOTE_CAST").show(10, truncate=False)

Số dòng hợp lệ: 42893
Số dòng không hợp lệ: 1229
+------+-------------------+---------+
|ROW_ID|VOTER_NAME         |VOTE_CAST|
+------+-------------------+---------+
|100   |Lee M. Kleinman    |NO       |
|140   |Scott  Griggs      |NO       |
|299   |Scott  Griggs      |NO       |
|302   |Sandy  Greyson     |NO       |
|303   |Jennifer S.  Gates |NO       |
|325   |Michael S. Rawlings|ABSNT_CB |
|356   |Adam Medrano       |ABST     |
|365   |Lee M. Kleinman    |ABST     |
|370   |Adam Medrano       |ABST     |
|425   |Lee M. Kleinman    |NO       |
+------+-------------------+---------+
only showing top 10 rows


## 5. Câu hỏi ôn tập

1. Vì sao schema tường minh có thể cải thiện hiệu năng đọc dữ liệu?
2. Immutable DataFrame có ý nghĩa gì?
3. Transformation khác action như thế nào?
4. Vì sao Parquet phù hợp cho phân tích dữ liệu lớn?
5. Khi nào nên dùng UDF? Khi nào không nên dùng?
6. monotonically_increasing_id() có tạo ID liên tục hay không?
7. Cache có phải lúc nào cũng làm chương trình nhanh hơn không?
8. Shuffle là gì và tại sao cần hạn chế?
9. Broadcast join thích hợp trong trường hợp nào?
10. left_anti join hữu ích như thế nào trong data validation?

**1. Vì sao schema tường minh có thể cải thiện hiệu năng đọc dữ liệu?**

Schema tường minh (explicit schema) cho phép chúng ta xác định trước tên cột và kiểu dữ liệu của từng cột. Nhờ đó, Spark không cần hoặc giảm suy luận kiểu dữ liệu (schema inference), giúp tiết kiệm thời gian và tài nguyên khi đọc dữ liệu, đặc biệt với các tệp lớn.


**2. Immutable DataFrame có ý nghĩa gì?**

Immutable DataFrame có nghĩa là DataFrame không thể bị thay đổi trực tiếp sau khi được tạo. Khi sử dụng các phép biến đổi như `withColumn()`, `filter()` hoặc `select()`, Spark trả về một DataFrame mới thay vì sửa đổi DataFrame ban đầu.

**3. Transformation khác action như thế nào?**

| Tiêu chí  | Transformation                      | Action                                                 |
| --------- | ----------------------------------- | ------------------------------------------------------ |
| Chức năng | Xác định cách biến đổi dữ liệu      | Yêu cầu Spark thực thi để trả kết quả hoặc ghi dữ liệu |
| Kết quả   | Thường trả về DataFrame, RDD mới    | Trả kết quả về driver hoặc ghi ra nơi lưu trữ          |
| Ví dụ     | `filter()`, `select()`, `groupBy()` | `count()`, `collect()`, `show()`, `write()`            |

Spark sử dụng cơ chế lazy evaluation: các transformation thường chưa thực thi ngay mà xây dựng kế hoạch tính toán. Khi action được gọi, Spark mới thực thi các bước cần thiết để tạo ra kết quả.


**4. Vì sao Parquet phù hợp cho phân tích dữ liệu lớn?**

Parquet là định dạng lưu trữ dạng cột (columnar storage), phù hợp với các truy vấn phân tích thường chỉ cần một số cột trong bảng dữ liệu lớn.

Các ưu điểm chính:

- Đọc chọn lọc cột: Spark có thể chỉ đọc những cột cần thiết thay vì toàn bộ dữ liệu.
- Nén hiệu quả: Các giá trị cùng kiểu dữ liệu thường có thể được nén tốt.
- Lưu kiểu dữ liệu: Schema được lưu cùng dữ liệu, giúp hạn chế việc suy luận kiểu dữ liệu khi đọc.
- Tối ưu truy vấn: Hỗ trợ các kỹ thuật như column pruning và predicate pushdown khi điều kiện phù hợp.


**5. Khi nào nên dùng UDF? Khi nào không nên dùng?**

UDF (User-Defined Function) là hàm do người dùng tự định nghĩa để xử lý những logic mà các hàm tích hợp sẵn của Spark không đáp ứng thuận tiện.

Nên dùng UDF khi:

- Cần áp dụng logic nghiệp vụ đặc thù.
- Không thể biểu diễn phép tính bằng các hàm có sẵn của Spark SQL hoặc PySpark.

Không nên dùng UDF khi có thể thay thế bằng các hàm tích hợp như `when()`, `regexp_replace()`, `split()` hoặc `lower()`.

**6. `monotonically_increasing_id()` có tạo ID liên tục hay không?**

Không. Hàm `monotonically_increasing_id()` tạo các ID tăng theo thứ tự trong phạm vi thực thi, nhưng không đảm bảo các ID liên tiếp nhau.

**7. Cache có phải lúc nào cũng làm chương trình nhanh hơn không?**

Không. `cache()` lưu dữ liệu đã tính toán để tái sử dụng, giúp tránh thực hiện lại những phép tính tốn kém khi cùng một DataFrame được sử dụng nhiều lần. Nếu DataFrame chỉ được sử dụng một lần, chi phí cache có thể lớn hơn lợi ích.

**8. Shuffle là gì và tại sao cần hạn chế?**

Shuffle là quá trình phân phối lại dữ liệu giữa các phân vùng hoặc các máy trong cụm Spark để phục vụ những phép tính cần dữ liệu được nhóm hoặc kết hợp theo khóa.

Shuffle thường xuất hiện trong các thao tác như:

- `groupBy()` và `groupByKey()`
- `distinct()`
- `repartition()`
- Các phép join cần phân phối lại dữ liệu.

Shuffle có thể tốn kém vì phải truyền dữ liệu qua mạng, ghi/đọc dữ liệu trung gian và sử dụng tài nguyên CPU, bộ nhớ hoặc ổ đĩa.

Để hạn chế chi phí shuffle, có thể lọc dữ liệu sớm, chỉ chọn những cột cần thiết, sử dụng broadcast join khi thích hợp và tránh `repartition()` không cần thiết.

**9. Broadcast join thích hợp trong trường hợp nào?**

Broadcast join thích hợp khi một bảng tương đối nhỏ có thể được gửi đến các executor, trong khi bảng còn lại lớn hơn đáng kể.

**10. `left_anti` join hữu ích như thế nào trong data validation?**

`left_anti` join trả về các dòng của bảng bên trái không tìm thấy bản ghi khớp trong bảng bên phải theo điều kiện join.

Phép join này rất hữu ích để phát hiện dữ liệu không hợp lệ, khóa ngoại bị thiếu hoặc bản ghi chưa được ánh xạ.

## Tổng kết

Sau bài thực hành, sinh viên cần nắm được quy trình làm sạch dữ liệu lớn bằng PySpark, từ khai báo
schema và đọc dữ liệu đến cleaning, validation, tối ưu thực thi và xuất dữ liệu sạch.
Chuỗi xử lý nên ghi nhớ: Read → Validate → Clean → Transform → Analyze → Write